# Phát hiện tàu từ Google Drive — mô hình allenai/vessel-detection-sentinels

Notebook này làm **cùng một việc** với `infer_drive_folder_ship_detection.ipynb`
(nhận vào 1 **link thư mục Google Drive** + **đường dẫn mô hình**, chạy suy luận
trên ảnh Sentinel-2 rồi xuất kết quả), nhưng thay bộ phát hiện **YOLO (Ultralytics)**
bằng **Faster R-CNN + Swin Transformer V2** của
[`allenai/vessel-detection-sentinels`](https://github.com/allenai/vessel-detection-sentinels).

**Đầu ra** (theo yêu cầu: **không** xuất file `.tif` annotated — tải `.geojson` về QGIS):

| File | Nội dung |
|---|---|
| `{stem}_pred.geojson` / `.gpkg` | khung tàu **sau** lọc mây — kéo thẳng vào QGIS |
| `{stem}_pred_points.geojson` | cùng dữ liệu nhưng ở dạng **điểm** (mô hình vốn là point-detector) |
| `{stem}_pred_raw.geojson` / `.gpkg` | dự đoán **trước** lọc mây, để đối chiếu |
| `{stem}_predictions.csv` | 1 dòng / 1 phát hiện: `score`, `lon`, `lat`, cột/hàng ảnh, thuộc tính tàu |
| `{stem}_detections.json` | như trên, dạng JSON |
| `{stem}_overview.png` | ảnh thu nhỏ toàn cảnh có khung, để mắt thường kiểm tra |
| `{stem}_crops.png` / `{stem}_crops.zip` | ảnh cắt quanh từng phát hiện |
| `summary.csv` | tổng hợp cả lượt chạy |

GeoJSON luôn ở **EPSG:4326**, GeoPackage giữ **CRS gốc của ảnh** (thường là UTM).

## 0. Các "pretrain" của repo nằm ở đâu? (`torch_weights/` là gì)

Đã kiểm tra trực tiếp trong repo — có **hai loại trọng số hoàn toàn khác nhau**:

**1. `torch_weights/` — CHỈ là backbone ImageNet của torchvision, KHÔNG phải model dò tàu.**

Thư mục này có đúng 3 file, đều là checkpoint gốc của torchvision:
`swin_v2_t-b137f0e2.pth`, `swin_v2_s-637d8ceb.pth`, `resnet50-0676ba61.pth`.
Chúng **không được code Python nào đọc trực tiếp**; chỗ duy nhất nhắc đến chúng là `Dockerfile`:

```dockerfile
COPY torch_weights/swin_v2_s-637d8ceb.pth /root/.cache/torch/hub/checkpoints/...
COPY torch_weights/resnet50-0676ba61.pth  /root/.cache/torch/hub/checkpoints/...
COPY torch_weights/swin_v2_t-b137f0e2.pth /root/.cache/torch/hub/checkpoints/...
```

Tức là chúng chỉ được **nạp sẵn vào cache của `torch.hub`** để container chạy được khi
**không có mạng**. Lý do cần chúng: ngay cả lúc *suy luận*, code vẫn dựng backbone kèm
trọng số ImageNet trước khi nạp trọng số thật —

* `src/models/frcnn_cmp2.py:200` → `swin_v2_{t,s,b}(weights=Swin_V2_*_Weights.IMAGENET1K_V1)`
* `src/models/custom.py:14` → `resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)`

Ngay sau đó `src/inference/pipeline.py:272` gọi `model.load_state_dict(torch.load(best.pth))`,
**ghi đè toàn bộ** trọng số ImageNet đó. Nên: máy **có mạng** thì bỏ qua `torch_weights/` cũng
chạy bình thường (torchvision tự tải về cache); máy **offline** thì mới cần chúng.

**2. Trọng số THẬT của mô hình phát hiện tàu nằm ở `model_artifacts/`, mỗi mô hình = `cfg.json` + `best.pth`.**

`src/config/config.yml` trỏ tới các thư mục này. Những gì repo **thật sự kèm theo** (qua Git LFS):

| Mô hình | Đường dẫn | Có `best.pth`? |
|---|---|---|
| Detector Sentinel-1 | `frcnn_cmp2/3dff445` | ✅ có |
| Attr Sentinel-1 | `attr/c34aa37` | ✅ có |
| Attr Sentinel-2 | `attr/2feb47b-sentinel2-attr-resnet-tci-IR`, `attr/e609150-sentinel2-attr-resnet` | ✅ có |
| **Detector Sentinel-2** | `frcnn_cmp2/2feb47b-sentinel2-swinv2-small-tci-IR`, `frcnn_cmp2/15cddd5-sentinel2-swinv2-small-fpn` | ❌ **chỉ có `cfg.json`** |
| `multihead4/221e8ac9-...-satlas-weights` | mặc định của `config.yml` cho Sentinel-2 | ❌ chỉ `cfg.json`, **và kiến trúc `multihead4` không có trong `src/models/__init__.py`** → không chạy được từ repo này |

README nói thẳng: *"or by using pre-trained weights we will provide separately"* cho Sentinel-2.

**Hệ quả cho notebook này:** phải **tự cung cấp `best.pth`** của detector Sentinel-2 và trỏ
`CFG['DETECTOR_MODEL_DIR']` vào thư mục chứa cặp `cfg.json` + `best.pth` (tải lên Kaggle Dataset,
Drive, hoặc thư mục cục bộ). Ô nạp mô hình bên dưới sẽ báo lỗi rõ ràng nếu thiếu `best.pth`
hoặc nếu file mới chỉ là con trỏ Git LFS chưa tải.

Nếu chỉ muốn thử với **Sentinel-1** (weights có sẵn trong repo): đặt `REPO_LFS = True`,
`CATALOG = "sentinel1"` và `DETECTOR_MODEL_DIR = "frcnn_cmp2/3dff445"`.

## 1. Config

Chỉ cần sửa 2 dòng: `DRIVE_FOLDER_URL` và `DETECTOR_MODEL_DIR`.

In [ ]:
import os

CFG = dict(
    # ---------------- Nguồn ảnh: link thư mục Google Drive ----------------
    DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/12QVIJ3H4_h_YeKHEJ8U2387wuMKRd4Qz",  # <<< SỬA
    LIMIT = None,                  # None = chạy hết; hoặc số ảnh tối đa

    # ---------------- Mô hình: artifact của allenai/vessel-detection-sentinels ----------------
    # MỖI thư mục artifact phải có ĐỦ 2 file:  cfg.json  +  best.pth
    #   cfg.json  -> kiến trúc (Architecture/Options/Channels)
    #   best.pth  -> TRỌNG SỐ ĐÃ HUẤN LUYỆN của mô hình phát hiện tàu
    # Repo KHÔNG kèm weights của detector Sentinel-2 (chỉ có cfg.json) -> phải tự trỏ tới
    # thư mục weights của bạn (Kaggle Dataset / Drive / ổ đĩa). Xem mục 0 ở trên.
    DETECTOR_MODEL_DIR = "/kaggle/input/vessel-detection-sentinels/frcnn_cmp2/2feb47b-sentinel2-swinv2-small-tci-IR",  # <<< SỬA
    POSTPROCESS_MODEL_DIR = "",    # "" = bỏ qua dự đoán thuộc tính (dài/rộng/hướng/tốc độ/nghề cá)
    CATALOG = "sentinel2",         # "sentinel2" | "sentinel1"

    # Repo chỉ được clone để LẤY MÃ NGUỒN mô hình (src/models). Không cài requirements.txt
    # của repo (pin torch 1.13 / gdal 3.4) để khỏi phá môi trường Kaggle.
    REPO_URL = "https://github.com/allenai/vessel-detection-sentinels.git",
    REPO_DIR = "/kaggle/working/vessel-detection-sentinels",
    REPO_LFS = False,              # True = kéo luôn file .pth trong repo qua Git LFS (~1GB, chỉ có S1 + attr)

    # ---------------- Tham số suy luận (mặc định theo src/main.py của repo) ----------------
    WINDOW  = 2048,                # cạnh cửa sổ trượt (px, trên ảnh đã warp về EPSG:3857)
    PADDING = 400,                 # viền bỏ qua ở mỗi cạnh cửa sổ (chống cắt đôi tàu)
    OVERLAP = 20,                  # nới thêm biên giữ lại quanh vùng padding
    CONF    = 0.9,                 # ngưỡng điểm tin cậy của Faster R-CNN (repo dùng 0.9)
    NMS_DIST = 10,                 # NMS theo KHOẢNG CÁCH tâm (px) — đúng cách repo làm
    NMS_IOU  = 0.5,                # NMS theo IoU chạy trước (0 = tắt)
    BATCH   = 1,                   # số cửa sổ mỗi lượt forward (repo chạy 1; tăng nếu GPU khoẻ)
    SKIP_DARK = 5,                 # bỏ cửa sổ toàn đen/nodata (max < ngưỡng)
    DEVICE  = "auto",
    S2_PIXEL_SIZE = 10.0,          # GSD nguồn (m) dùng để chọn mức zoom khi warp; repo hardcode 10
    MISSING_BAND = "zeros",        # thiếu band mà model cần (vd B08): "zeros" = đắp 0 | "skip" = bỏ ảnh

    # ---------------- Mask mây (Sentinel-2 L1C/L2A) ----------------
    # Giống notebook YOLO: dùng mask mây của Copernicus để loại dự đoán rơi vào vùng mây.
    # LƯU Ý: notebook này chỉ hỗ trợ chế độ "after" (lọc SAU khi infer), vì ảnh được
    # infer trên lưới EPSG:3857 còn mask mây nằm trên lưới UTM gốc.
    CLOUD_MASK = True,
    CLOUD_MASK_MODE = "after",     # "after" | "off"
    CLOUD_SOURCE = "auto",         # "auto" (SCL/MSK_CLASSI/độ sáng) | "omnicloudmask"
    CLOUD_OCM_CLASSES = (1, 2),
    CLOUD_OCM_RES = 20.0,
    CLOUD_OCM_BATCH = 1,
    CLOUD_OCM_DEVICE = None,
    CLOUD_FILTER = "fraction",     # "fraction" (theo % diện tích) | "center"
    CLOUD_FRAC_THR = 0.35,
    # 0 = tắt bảo vệ. Khác YOLO: ở đây CONF đã là 0.9 nên gần như mọi dự đoán đều
    # "score cao", đặt ngưỡng bảo vệ sẽ vô hiệu hoá luôn việc lọc mây. Lọc bằng B10
    # bên dưới vốn không phụ thuộc score nên vẫn giữ được tàu thật.
    CLOUD_PROTECT_CONF = 0.0,
    CLOUD_B10 = True,              # (L1C) bỏ box sáng ở B10 1375nm = cirrus
    CLOUD_B10_THR = 0.010,
    CLOUD_B10_BG_PCT = 20,
    CLOUD_SCL_CLASSES = (8, 9, 10),
    CLOUD_INCLUDE_SHADOW = False,
    CLOUD_INCLUDE_SNOW = False,
    CLOUD_CLDPRB_THR = 40,
    CLOUD_DILATE_PX = 0,
    CLOUD_SKIP_IF_TCI_ONLY = True,
    CLOUD_BRIGHT_THR = 0.60,
    CLOUD_SAT_THR = 0.20,
    DRAW_CLOUD = True,

    # ---------------- Đầu ra (KHÔNG xuất GeoTIFF/JP2 annotated) ----------------
    SAVE_VECTOR = True,            # {stem}_pred.geojson + .gpkg  -> kéo thẳng vào QGIS
    SAVE_VECTOR_RAW = True,        # {stem}_pred_raw.geojson/.gpkg = TRƯỚC lọc mây
    SAVE_POINTS = True,            # xuất thêm lớp ĐIỂM {stem}_pred_points.geojson (mô hình vốn là point-detector)
    SAVE_CSV = True,               # {stem}_predictions.csv (giống predictions.csv của repo)
    SAVE_JSON = True,              # {stem}_detections.json
    SAVE_OVERVIEW = True,          # {stem}_overview.png để mắt thường kiểm tra nhanh
    OVERVIEW_MAX = 1600,
    SAVE_CROPS_PNG = True,         # {stem}_crops.png (lưới top-k)
    SAVE_CROPS_ZIP = True,         # {stem}_crops.zip (mỗi phát hiện 1 ảnh)
    CROPS_PAD = 40,
    CROPS_DRAW_BOX = True,
    SAVE_HARD_NEG = False,         # tile hard-negative (định dạng YOLO) để fine-tune lại model khác
    HARD_NEG_SOURCE = "dropped",   # "dropped" | "kept" | "all"
    HARD_NEG_TILE = 800,

    UNZIP_DELETE_ZIP = False,
    DL_DIR  = "/kaggle/working/drive_tci",
    OUT_DIR = "/kaggle/working/drive_infer_vds",
)

os.makedirs(CFG["DL_DIR"], exist_ok=True)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
print("Detector dir :", CFG["DETECTOR_MODEL_DIR"])
print("Attr  dir    :", CFG["POSTPROCESS_MODEL_DIR"] or "(tắt)")
print("Conf         :", CFG["CONF"], "| window:", CFG["WINDOW"], "| padding:", CFG["PADDING"])
print("Cloud masking:", CFG["CLOUD_MASK"], "| mode:", CFG["CLOUD_MASK_MODE"])

## 2. Thư viện & mã nguồn mô hình

In [ ]:
%pip install -q -U "huggingface_hub>=0.24" rasterio geopandas gdown pyproj omnicloudmask

import os, subprocess, sys

# --- Clone repo allenai/vessel-detection-sentinels: CHỈ để dùng mã mô hình (src/models) ---
# KHÔNG chạy `pip install -r requirements.txt` của repo: nó ghim torch==1.13 / gdal==3.4 và
# sẽ phá môi trường Kaggle. Mã trong src/models chỉ cần torch + torchvision.
if not os.path.isdir(os.path.join(CFG["REPO_DIR"], "src")):
    env = dict(os.environ)
    if not CFG.get("REPO_LFS", False):
        env["GIT_LFS_SKIP_SMUDGE"] = "1"   # bỏ qua file .pth nặng trong repo
    subprocess.run(["git", "clone", "--depth", "1", CFG["REPO_URL"], CFG["REPO_DIR"]],
                   check=True, env=env)
else:
    print("Repo đã có sẵn:", CFG["REPO_DIR"])

if CFG["REPO_DIR"] not in sys.path:
    sys.path.insert(0, CFG["REPO_DIR"])   # để `import src.models` chạy được

import torch, torchvision
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)

import rasterio, geopandas
_jp2 = rasterio.drivers.raster_driver_extensions().get("jp2", "")
print("rasterio:", rasterio.__version__, "| driver JP2:", _jp2 or "(không có -> không đọc được .jp2!)")


def pick_device(pref):
    if pref not in ("auto", None, ""):
        print("device preference:", pref)
        return torch.device(str(pref))
    if not torch.cuda.is_available():
        print("cuda not available -> cpu (Faster R-CNN trên CPU rất chậm)")
        return torch.device("cpu")
    try:
        _ = (torch.zeros(16, device="cuda") + 1).sum().item()
        torch.cuda.synchronize()
        print("gpu detected:", torch.cuda.get_device_name(0))
        return torch.device("cuda")
    except Exception as e:
        print("Not found GPU kernel:", str(e).splitlines()[0])
        return torch.device("cpu")


DEVICE = pick_device(CFG["DEVICE"])
print("device used =", DEVICE)

## 3. Nạp mô hình từ thư mục artifact (`cfg.json` + `best.pth`)

In [ ]:
import glob
import json
import torch

from src.models import models as VDS_ARCHS   # dict: frcnn / frcnn_cmp2 / custom / resnet / unet


class Channels:
    """Bản rút gọn của `src.data.image.Channels`.

    Không import trực tiếp src/data/image.py vì file đó kéo theo `osgeo.gdal`
    (không có sẵn trên Kaggle). Mô hình chỉ dùng `.count()`.
    """

    def __init__(self, channels):
        self.channels = list(channels)

    def __len__(self):
        return len(self.channels)

    def __getitem__(self, idx):
        return self.channels[idx]

    def count(self):
        return sum(int(c["Count"]) for c in self.channels)

    def flatten(self):
        out = []
        for c in self.channels:
            if int(c["Count"]) > 1:
                out += [f"{c['Name']}-{i}" for i in range(int(c["Count"]))]
            else:
                out.append(c["Name"])
        return out

    def with_ranges(self):
        out, cur = [], 0
        for c in self.channels:
            out.append((c, (cur, cur + int(c["Count"]))))
            cur += int(c["Count"])
        return out


def read_model_cfg(model_dir):
    with open(os.path.join(model_dir, "cfg.json"), "r") as f:
        return json.load(f)


def base_channel_names(model_cfg):
    """Tên các kênh KHÔNG phải overlap, giữ nguyên thứ tự. Ví dụ -> ['tci', 'b08'].

    Repo xếp mảng đầu vào theo NHÓM ẢNH: [tci, b08] của ảnh chính, rồi [tci, b08]
    của overlap-1, overlap-2... (xem src/data/image.py::prepare_scenes), đúng bằng
    `Options.GroupChannels`. Notebook này chạy KHÔNG dùng ảnh lịch sử nên chỉ đưa
    vào nhóm đầu; backbone tự đắp "blank history" khi thiếu overlap
    (src/models/frcnn_cmp2.py::SwinTransformerRCNNBackbone.forward).
    """
    seen, out = set(), []
    for c in model_cfg["Channels"]:
        name = c["Name"]
        if "overlap" in name or name in seen:
            continue
        seen.add(name)
        out.append(dict(Name=name, Count=int(c["Count"])))
    return out


def check_weights(model_dir):
    """Kiểm tra best.pth có thật (và không phải con trỏ Git LFS chưa tải)."""
    path = os.path.join(model_dir, "best.pth")
    if not os.path.isdir(model_dir):
        raise FileNotFoundError(f"Không thấy thư mục model: {model_dir}")
    if not os.path.exists(os.path.join(model_dir, "cfg.json")):
        raise FileNotFoundError(f"Thiếu cfg.json trong {model_dir}")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Thiếu best.pth trong {model_dir}.\n"
            "Repo allenai/vessel-detection-sentinels KHÔNG kèm trọng số detector Sentinel-2 "
            "(chỉ có cfg.json). Hãy trỏ DETECTOR_MODEL_DIR tới thư mục weights của bạn."
        )
    with open(path, "rb") as f:
        head = f.read(64)
    if head.startswith(b"version https://git-lfs"):
        raise RuntimeError(
            f"{path} mới chỉ là con trỏ Git LFS. Chạy `git lfs install && git lfs pull` "
            "trong repo, hoặc đặt CFG['REPO_LFS']=True rồi clone lại."
        )
    return path


def load_artifact_model(model_dir, example_chw, device):
    """Dựng mô hình từ cfg.json rồi nạp best.pth — y hệt src/inference/pipeline.py::load_model."""
    weights_path = check_weights(model_dir)
    model_cfg = read_model_cfg(model_dir)
    arch = model_cfg["Architecture"]
    if arch not in VDS_ARCHS:
        raise KeyError(
            f"Kiến trúc '{arch}' không có trong src/models/__init__.py "
            f"(chỉ có: {sorted(VDS_ARCHS)}). Ví dụ 'multihead4' được nhắc trong "
            "src/config/config.yml nhưng KHÔNG có mã nguồn trong repo."
        )
    # LƯU Ý: truyền model_cfg['Data'] NGUYÊN VẸN. Trong repo, `categories` là một CHUỖI
    # '["vessel"]' (đọc thẳng từ cột TEXT của metadata.sqlite3), nên
    # num_classes = len('["vessel"]') = 10 cả lúc train lẫn lúc infer. Nếu json.loads()
    # nó thành list 1 phần tử thì đầu ra box_predictor sẽ lệch shape với best.pth.
    model = VDS_ARCHS[arch]({
        "Channels": Channels(model_cfg["Channels"]),
        "Device": device,
        "Model": model_cfg,
        "Options": model_cfg["Options"],
        "Data": model_cfg["Data"],
        "Example": [example_chw, None],
    })
    state = torch.load(weights_path, map_location=device, weights_only=False)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print(f"[model] {arch} <- {model_dir}")
    print(f"        channels={Channels(model_cfg['Channels']).count()} "
          f"base={[c['Name'] for c in base_channel_names(model_cfg)]} "
          f"ImageSize(train)={model_cfg['Options'].get('ImageSize')}")
    return model, model_cfg


def resolve_model_dir(path, repo_dir):
    """Cho phép ghi đường dẫn tương đối trong repo, vd 'frcnn_cmp2/3dff445'."""
    if not path:
        return ""
    if os.path.isdir(path):
        return path
    for root in (os.path.join(repo_dir, "src", "model_artifacts"),
                 os.path.join(repo_dir, "data", "model_artifacts"),
                 os.path.join(repo_dir, "data", "model_artifacts", "sentinel-1"),
                 os.path.join(repo_dir, "data", "model_artifacts", "sentinel-2")):
        cand = os.path.join(root, path)
        if os.path.isdir(cand):
            return cand
    hits = sorted(glob.glob(os.path.join(repo_dir, "**", path), recursive=True))
    return hits[0] if hits else path


DETECTOR_DIR = resolve_model_dir(CFG["DETECTOR_MODEL_DIR"], CFG["REPO_DIR"])
ATTR_DIR = resolve_model_dir(CFG["POSTPROCESS_MODEL_DIR"], CFG["REPO_DIR"])

DET_CFG = read_model_cfg(DETECTOR_DIR)
DET_CHANNELS = base_channel_names(DET_CFG)                 # [{'Name': 'tci', 'Count': 3}, ...]
GROUP_CHANNELS = int(DET_CFG["Options"].get("GroupChannels", sum(c["Count"] for c in DET_CHANNELS)))
assert GROUP_CHANNELS == sum(c["Count"] for c in DET_CHANNELS), (
    f"GroupChannels={GROUP_CHANNELS} khác tổng kênh gốc={sum(c['Count'] for c in DET_CHANNELS)}")

_example = torch.zeros((GROUP_CHANNELS, CFG["WINDOW"], CFG["WINDOW"]), dtype=torch.float32)
detector, _ = load_artifact_model(DETECTOR_DIR, _example, DEVICE)

attr_model = None
if ATTR_DIR:
    _ex_attr = torch.zeros((GROUP_CHANNELS, 120, 120), dtype=torch.float32)
    attr_model, _ = load_artifact_model(ATTR_DIR, _ex_attr, DEVICE)
del _example

## 4. Mask mây (Sentinel-2 L1C / L2A)

Giữ nguyên từ notebook YOLO. Mô hình hay nhận nhầm **cụm mây nhỏ, sáng** thành tàu,
nên ta dùng mask mây của Copernicus để loại các dự đoán rơi vào vùng mây.

- **L2A**: lớp phân loại cảnh **SCL** (8=mây vừa, 9=mây cao, 10=cirrus), hoặc `MSK_CLDPRB` / `MSK_CLASSI`.
- **L1C**: `MSK_CLASSI` (baseline ≥ 04.00) hoặc `MSK_CLOUDS` (GML, baseline cũ), cộng thêm lọc cirrus bằng band **B10 (1375nm)**.
- **Chỉ có ảnh TCI** (không có band/mask kèm theo): mặc định **bỏ qua lọc mây** cho ảnh đó.

Notebook này chỉ dùng chế độ **`after`** (suy luận xong mới lọc), vì ảnh được suy luận trên
lưới EPSG:3857 còn mask mây nằm trên lưới UTM gốc — box được đưa **về pixel ảnh gốc** trước
khi lọc nên hai lưới khớp nhau.

In [ ]:
# =============================================================================
#  Mask mây cho Sentinel-2 (L1C / L2A)
# -----------------------------------------------------------------------------
#  - Nhận dạng mức xử lý sản phẩm (L1C hay L2A) từ TÊN FILE; nếu tên không chuẩn
#    thì đọc METADATA (MTD_MSIL1C.xml / MTD_MSIL2A.xml / MTD_TL.xml) hoặc suy ra
#    từ cấu trúc thư mục (có SCL / R10m,R20m,R60m => L2A).
#  - Tự tìm nguồn mask mây tốt nhất trong folder Copernicus (.SAFE):
#       L2A: SCL  >  MSK_CLDPRB  >  MSK_CLASSI
#       L1C: MSK_CLASSI (baseline >= 04.00)  >  MSK_CLOUDS (GML, baseline cũ)
#       Chỉ có TCI: fallback theo độ sáng/độ bão hoà.
#  - "after"  (khuyến nghị): infer trên ảnh gốc, sau đó BỎ các box trùng vùng mây.
#  - "before" : xoá (đen hoá) pixel mây trước khi infer.
# =============================================================================
import os, re, glob
import numpy as np
import cv2
import rasterio
from rasterio.enums import Resampling
from affine import Affine

# ---- tên file Sentinel-2 -----------------------------------------------------
_S2_BAND_RE = re.compile(r"_(B0[1-9]|B1[0-2]|B8A|AOT|WVP|SCL|PVI|TCI)_?\d*m?\.(jp2|tif|tiff)$", re.I)
_S2_MASK_RE = re.compile(r"MSK_[A-Z0-9]+", re.I)
_TCI_RE     = re.compile(r"_TCI(_\d{2}m)?\.(jp2|tif|tiff)$", re.I)


def is_tci(path):
    return bool(_TCI_RE.search(os.path.basename(path)))


def is_s2_band_or_mask(path):
    """True cho các file band/phụ trợ Sentinel-2 (B01..B12, SCL, MSK_*, PVI...)
    — đây là đầu vào để tính mask, KHÔNG phải ảnh để chạy detect. TCI được loại
    khỏi nhóm này vì TCI chính là ảnh ta infer."""
    if is_tci(path):
        return False
    b = os.path.basename(path)
    return bool(_S2_BAND_RE.search(b) or _S2_MASK_RE.search(b))


# ---- L1C vs L2A --------------------------------------------------------------
def find_safe_root(path):
    """Đi ngược lên tới thư mục gốc sản phẩm: kết thúc bằng .SAFE, hoặc chứa
    MTD_MSIL1C.xml / MTD_MSIL2A.xml, hoặc chứa thư mục GRANULE."""
    d = path if os.path.isdir(path) else os.path.dirname(path)
    prev = None
    while d and d != prev:
        if d.upper().endswith(".SAFE"):
            return d
        try:
            entries = set(os.listdir(d))
        except OSError:
            entries = set()
        if {"MTD_MSIL1C.xml", "MTD_MSIL2A.xml"} & entries or "GRANULE" in entries:
            return d
        prev, d = d, os.path.dirname(d)
    return None


def _level_from_metadata(safe_root):
    cands = [os.path.join(safe_root, "MTD_MSIL1C.xml"),
             os.path.join(safe_root, "MTD_MSIL2A.xml")]
    cands += glob.glob(os.path.join(safe_root, "MTD_MSIL*.xml"))
    cands += glob.glob(os.path.join(safe_root, "GRANULE", "*", "MTD_TL.xml"))
    for xml in cands:
        if not os.path.exists(xml):
            continue
        try:
            up = open(xml, "r", errors="ignore").read(20000).upper()
        except OSError:
            continue
        if "S2MSI2A" in up or "LEVEL-2A" in up:
            return "L2A"
        if "S2MSI1C" in up or "LEVEL-1C" in up:
            return "L1C"
    return None


def _level_from_structure(safe_root):
    if glob.glob(os.path.join(safe_root, "GRANULE", "*", "IMG_DATA", "R*m")):
        return "L2A"
    if glob.glob(os.path.join(safe_root, "GRANULE", "*", "**", "*_SCL_*.jp2"), recursive=True):
        return "L2A"
    if glob.glob(os.path.join(safe_root, "GRANULE", "*", "IMG_DATA", "*_B0*.jp2")):
        return "L1C"
    return None


def detect_product_level(path, safe_root=None, verbose=False):
    """Trả 'L1C' | 'L2A' | None cho 1 raster hoặc folder sản phẩm.
    Thứ tự: (1) token trong tên file, (2) hậu tố _TCI_10m chỉ có ở L2A,
    (3) metadata XML, (4) cấu trúc thư mục. -> Trả lời câu hỏi: tên file không
    chuẩn vẫn xác định được mức nhờ (3)-(4)."""
    up = str(path).upper()
    if "MSIL2A" in up or "MSI_L2A" in up:
        lvl = "L2A"
    elif "MSIL1C" in up or "MSI_L1C" in up:
        lvl = "L1C"
    elif re.search(r"_TCI_\d{2}M", up):
        lvl = "L2A"
    else:
        root = safe_root or find_safe_root(path)
        lvl = (_level_from_metadata(root) or _level_from_structure(root)) if root else None
    if verbose:
        print(f"[cloud] product level for {os.path.basename(str(path))}: {lvl}")
    return lvl


# ---- tìm nguồn mask mây ------------------------------------------------------
def _granule_root(raster_path):
    d = os.path.dirname(raster_path)
    for _ in range(4):
        if os.path.isdir(os.path.join(d, "IMG_DATA")) or os.path.isdir(os.path.join(d, "QI_DATA")):
            return d
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    return None


def _log(kind, path, verbose):
    if verbose:
        print(f"[cloud] source = {kind}: {os.path.basename(path)}")
    return kind, path


def find_cloud_source(raster_path, level=None, verbose=False):
    """(kind, path) cho nguồn mask tốt nhất cạnh TCI.
    L2A: 'scl' > 'cldprb' > 'classi'; L1C: 'classi' > 'clouds_gml';
    không có metadata -> ('brightness', tci)."""
    level = level or detect_product_level(raster_path)
    g = _granule_root(raster_path)

    def first(*patterns):
        for pat in patterns:
            hit = sorted(glob.glob(pat, recursive=True))
            if hit:
                return hit[0]
        return None

    if g:
        img, qi = os.path.join(g, "IMG_DATA"), os.path.join(g, "QI_DATA")
        if level == "L2A":
            scl = first(os.path.join(img, "R20m", "*_SCL_20m.jp2"),
                        os.path.join(img, "R60m", "*_SCL_60m.jp2"),
                        os.path.join(img, "**", "*_SCL_*.jp2"))
            if scl:
                return _log("scl", scl, verbose)
            cldprb = first(os.path.join(qi, "MSK_CLDPRB_20m.jp2"),
                           os.path.join(qi, "MSK_CLDPRB_60m.jp2"),
                           os.path.join(qi, "MSK_CLDPRB*.jp2"))
            if cldprb:
                return _log("cldprb", cldprb, verbose)
        classi = first(os.path.join(qi, "MSK_CLASSI_B00.jp2"),
                       os.path.join(qi, "MSK_CLASSI*.jp2"))
        if classi:
            return _log("classi", classi, verbose)
        gml = first(os.path.join(qi, "MSK_CLOUDS_B00.gml"),
                    os.path.join(qi, "MSK_CLOUDS*.gml"))
        if gml:
            return _log("clouds_gml", gml, verbose)
    return _log("brightness", raster_path, verbose)


# ---- CloudMask: mask boolean trên lưới riêng (array + transform + crs) --------
class CloudMask:
    def __init__(self, mask, transform, crs=None, kind="unknown"):
        self.mask = np.ascontiguousarray(mask, dtype=bool)
        self.transform = transform
        self.crs = crs
        self.kind = kind

    @property
    def coverage(self):
        return float(self.mask.mean()) if self.mask.size else 0.0

    def box_cloud_fraction(self, boxes, tci_transform):
        """Tỉ lệ (0..1) diện tích mỗi box (px của TCI, [x1,y1,x2,y2]) bị mây phủ."""
        boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
        if len(boxes) == 0 or self.mask.size == 0:
            return np.zeros(len(boxes), np.float32)
        inv = ~self.transform
        Hm, Wm = self.mask.shape
        out = np.zeros(len(boxes), np.float32)
        for i, (x1, y1, x2, y2) in enumerate(boxes):
            (X1, Y1) = tci_transform * (x1, y1)
            (X2, Y2) = tci_transform * (x2, y2)
            (c1, r1) = inv * (X1, Y1)
            (c2, r2) = inv * (X2, Y2)
            cmin, cmax = sorted((c1, c2)); rmin, rmax = sorted((r1, r2))
            cs = max(0, int(np.floor(cmin))); ce = min(Wm, int(np.ceil(cmax)))
            rs = max(0, int(np.floor(rmin))); re_ = min(Hm, int(np.ceil(rmax)))
            if ce <= cs or re_ <= rs:
                cc = min(Wm - 1, max(0, int((cmin + cmax) / 2)))
                rr = min(Hm - 1, max(0, int((rmin + rmax) / 2)))
                out[i] = float(self.mask[rr, cc]); continue
            out[i] = float(self.mask[rs:re_, cs:ce].mean())
        return out

    def window_mask(self, tci_transform, ox, oy, tw, th):
        """Mask mây (th, tw) cho 1 cửa sổ TCI tại offset (ox, oy) — dùng cho 'before'."""
        inv = ~self.transform
        Hm, Wm = self.mask.shape
        (X0, Y0) = tci_transform * (ox, oy)
        (X1, Y1) = tci_transform * (ox + tw, oy + th)
        (c0, r0) = inv * (X0, Y0); (c1, r1) = inv * (X1, Y1)
        cs = max(0, int(np.floor(min(c0, c1)))); ce = min(Wm, int(np.ceil(max(c0, c1))))
        rs = max(0, int(np.floor(min(r0, r1)))); re_ = min(Hm, int(np.ceil(max(r0, r1))))
        if ce <= cs or re_ <= rs:
            return np.zeros((th, tw), bool)
        sub = self.mask[rs:re_, cs:ce].astype(np.uint8)
        return cv2.resize(sub, (tw, th), interpolation=cv2.INTER_NEAREST).astype(bool)


def _dilate(mask, px):
    if px <= 0:
        return mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * px + 1, 2 * px + 1))
    return cv2.dilate(mask.astype(np.uint8), k).astype(bool)


def _brightness_mask(tci_path, bright_thr, sat_thr, max_side=2048):
    with rasterio.open(tci_path) as src:
        W, H = src.width, src.height
        scale = min(1.0, max_side / max(W, H))
        ow, oh = max(1, int(W * scale)), max(1, int(H * scale))
        idx = [1, 2, 3] if src.count >= 3 else [1, 1, 1]
        arr = src.read(indexes=idx, out_shape=(3, oh, ow)).astype(np.float32)
        transform = src.transform * Affine.scale(W / ow, H / oh)
        crs = src.crs
    mx = arr.max() or 1.0
    rgb = arr / mx
    mn = rgb.min(axis=0); mxc = rgb.max(axis=0)
    bright = mxc >= bright_thr
    sat = (mxc - mn) / (mxc + 1e-6)
    return (bright & (sat <= sat_thr)), transform, crs


def _rasterize_gml(gml_path, raster_path, max_side=4096):
    import geopandas as gpd
    from rasterio.features import rasterize
    with rasterio.open(raster_path) as src:
        W, H, crs, T = src.width, src.height, src.crs, src.transform
    scale = min(1.0, max_side / max(W, H))
    ow, oh = max(1, int(W * scale)), max(1, int(H * scale))
    transform = T * Affine.scale(W / ow, H / oh)
    try:
        gdf = gpd.read_file(gml_path)
        if crs is not None and gdf.crs is not None and str(gdf.crs) != str(crs):
            gdf = gdf.to_crs(crs)
        geoms = [g for g in gdf.geometry if g is not None and not g.is_empty]
    except Exception:
        geoms = []
    if not geoms:
        return np.zeros((oh, ow), bool), transform, crs
    m = rasterize([(g, 1) for g in geoms], out_shape=(oh, ow),
                  transform=transform, fill=0, dtype="uint8")
    return m.astype(bool), transform, crs


# ---- OmniCloudMask (deep learning; cần band B04/B03/B8A trong .SAFE) ----------
def _find_s2_band(granule, band, level):
    """Tìm file 1 band Sentinel-2 (vd 'B04') trong 1 granule .SAFE."""
    img = os.path.join(granule, "IMG_DATA")
    if level == "L2A":
        for res in ("R10m", "R20m", "R60m"):
            hit = sorted(glob.glob(os.path.join(img, res, f"*_{band}_*m.jp2")))
            if hit:
                return hit[0]
        hit = sorted(glob.glob(os.path.join(img, "**", f"*_{band}_*.jp2"), recursive=True))
        return hit[0] if hit else None
    hit = sorted(glob.glob(os.path.join(img, f"*_{band}.jp2")))
    if not hit:
        hit = sorted(glob.glob(os.path.join(img, "**", f"*_{band}.jp2"), recursive=True))
    return hit[0] if hit else None


def _omnicloudmask_mask(raster_path, cfg, verbose=True):
    """Mask mây bằng OmniCloudMask (CNN). Đọc Red=B04, Green=B03, NIR=B8A ở dạng
    raw DN (đúng như loader load_s2 của thư viện), suy luận, rồi lấy các lớp mây
    (1=dày, 2=mỏng) và tuỳ chọn bóng mây (3). Trả (mask, transform, crs)."""
    from omnicloudmask import predict_from_array
    level = detect_product_level(raster_path)
    g = _granule_root(raster_path)
    if g is None:
        raise RuntimeError("cần folder .SAFE để đọc band B04/B03/B8A, không chỉ ảnh TCI")
    red = _find_s2_band(g, "B04", level)
    green = _find_s2_band(g, "B03", level)
    nir = _find_s2_band(g, "B8A", level) or _find_s2_band(g, "B08", level)
    if not (red and green and nir):
        raise RuntimeError(f"thiếu band (B04={bool(red)} B03={bool(green)} NIR={bool(nir)})")
    target_res = float(cfg.get("CLOUD_OCM_RES", 20.0))
    with rasterio.open(red) as src:
        natW, natH = src.width, src.height
        nat_res = abs(src.transform.a) or 10.0
        crs, base_T = src.crs, src.transform
    factor = max(1.0, target_res / nat_res)
    outW = max(1, int(round(natW / factor)))
    outH = max(1, int(round(natH / factor)))
    transform = base_T * Affine.scale(natW / outW, natH / outH)

    def _rd(p):
        with rasterio.open(p) as s:
            return s.read(1, out_shape=(outH, outW), resampling=Resampling.bilinear).astype(np.float32)

    arr = np.stack([_rd(red), _rd(green), _rd(nir)], 0)   # (3,H,W) Red,Green,NIR raw DN
    dev = cfg.get("CLOUD_OCM_DEVICE")
    if not dev:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
    if verbose:
        print(f"[cloud] omnicloudmask {outW}x{outH}px @~{target_res:.0f}m device={dev} "
              f"(R={os.path.basename(red)}, NIR={os.path.basename(nir)})")
    pred = predict_from_array(arr, inference_device=dev, mosaic_device=dev,
                              batch_size=int(cfg.get("CLOUD_OCM_BATCH", 1)))
    labels = pred[0] if getattr(pred, "ndim", 2) == 3 else pred
    classes = set(cfg.get("CLOUD_OCM_CLASSES", (1, 2)))   # 1=mây dày, 2=mây mỏng
    if cfg.get("CLOUD_INCLUDE_SHADOW"):
        classes |= {3}                                    # 3=bóng mây
    mask = np.isin(labels, list(classes))
    return mask, transform, crs


def build_cloud_mask(raster_path, cfg=None, kind=None, source_path=None, verbose=True):
    """Dựng CloudMask cho raster_path; tự tìm nguồn nếu chưa cho. cfg=CFG để lấy ngưỡng."""
    cfg = cfg or {}
    if kind is None and str(cfg.get("CLOUD_SOURCE", "auto")).lower() == "omnicloudmask":
        try:
            mask, transform, crs = _omnicloudmask_mask(raster_path, cfg, verbose=verbose)
            mask = _dilate(mask, cfg.get("CLOUD_DILATE_PX", 0))
            cm = CloudMask(mask, transform, crs, kind="omnicloudmask")
            if verbose:
                print(f"[cloud] mask kind=omnicloudmask shape={cm.mask.shape} "
                      f"coverage={cm.coverage*100:.1f}%")
            return cm
        except Exception as e:
            print("[cloud] omnicloudmask lỗi -> quay về nguồn auto:", str(e).splitlines()[0])
    if kind is None:
        kind, source_path = find_cloud_source(raster_path, verbose=verbose)
    source_path = source_path or raster_path

    with rasterio.open(source_path) as src:
        transform, crs = src.transform, src.crs
        if kind == "scl":
            scl = src.read(1)
            classes = set(cfg.get("CLOUD_SCL_CLASSES", (8, 9, 10)))
            if cfg.get("CLOUD_INCLUDE_SHADOW"): classes |= {3}
            if cfg.get("CLOUD_INCLUDE_SNOW"):   classes |= {11}
            mask = np.isin(scl, list(classes))
        elif kind == "cldprb":
            mask = src.read(1).astype(np.float32) >= cfg.get("CLOUD_CLDPRB_THR", 40)
        elif kind == "classi":
            a = src.read()
            bands = a[:2] if a.shape[0] >= 2 else a
            mask = (bands > 0).any(axis=0)
        elif kind == "clouds_gml":
            mask, transform, crs = _rasterize_gml(source_path, raster_path)
        else:
            mask, transform, crs = _brightness_mask(
                source_path, cfg.get("CLOUD_BRIGHT_THR", 0.60), cfg.get("CLOUD_SAT_THR", 0.20))

    mask = _dilate(mask, cfg.get("CLOUD_DILATE_PX", 0))
    cm = CloudMask(mask, transform, crs, kind=kind)
    if verbose:
        print(f"[cloud] mask kind={kind} shape={cm.mask.shape} coverage={cm.coverage*100:.1f}%")
    return cm


def filter_boxes_by_cloud(boxes, scores, cloud_mask, tci_transform, frac_thr=0.35,
                          mode="fraction", protect_conf=0.0):
    """Tách detections thành giữ / bỏ theo mức trùng mây.
    mode='fraction': bỏ nếu >= frac_thr diện tích box là mây; 'center': bỏ nếu tâm box là mây.
    protect_conf>0: KHÔNG loại các dự đoán có conf >= protect_conf (bảo vệ tàu thật,
    tránh mất tàu khi mask mây quá mạnh tay / gán nhầm thân tàu là mây)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    if len(boxes) == 0:
        z = np.zeros((0, 4), np.float32)
        return z, scores[:0], z, scores[:0]
    if mode == "center":
        cx = (boxes[:, 0] + boxes[:, 2]) / 2; cy = (boxes[:, 1] + boxes[:, 3]) / 2
        frac = cloud_mask.box_cloud_fraction(np.stack([cx, cy, cx, cy], 1), tci_transform)
    else:
        frac = cloud_mask.box_cloud_fraction(boxes, tci_transform)
    drop = frac >= frac_thr
    if protect_conf and protect_conf > 0:
        drop = drop & (scores < float(protect_conf))
    return boxes[~drop], scores[~drop], boxes[drop], scores[drop]


# ---- Lọc cirrus bằng band B10 (1375nm) — CHỈ L1C (L2A đã bỏ B10) -------------
def _box_mean_values(values, val_transform, boxes, tci_transform):
    """Giá trị trung bình của 'values' (lưới riêng) dưới mỗi box (px của TCI)."""
    boxes = np.asarray(boxes, np.float64).reshape(-1, 4)
    out = np.zeros(len(boxes), np.float32)
    if len(boxes) == 0 or values.size == 0:
        return out
    inv = ~val_transform
    H, W = values.shape
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        (X1, Y1) = tci_transform * (x1, y1)
        (X2, Y2) = tci_transform * (x2, y2)
        (c1, r1) = inv * (X1, Y1)
        (c2, r2) = inv * (X2, Y2)
        cs = max(0, int(np.floor(min(c1, c2)))); ce = min(W, int(np.ceil(max(c1, c2))))
        rs = max(0, int(np.floor(min(r1, r2)))); re_ = min(H, int(np.ceil(max(r1, r2))))
        if ce <= cs or re_ <= rs:
            cc = min(W - 1, max(0, int((c1 + c2) / 2)))
            rr = min(H - 1, max(0, int((r1 + r2) / 2)))
            out[i] = values[rr, cc]
        else:
            out[i] = values[rs:re_, cs:ce].mean()
    return out


def filter_boxes_by_b10_cirrus(raster_path, boxes, scores, cfg, verbose=True):
    """Bỏ các box là CIRRUS/mây dựa trên band B10 (1375nm) của L1C.
    Cơ sở vật lý: hơi nước tầng thấp hấp thụ 1375nm nên mặt biển & tàu gần như TỐI
    ở B10, còn mây/cirrus (trên cao) thì SÁNG. -> lọc theo B10 an toàn cho tàu và
    KHÔNG phụ thuộc độ tin cậy (bắt được cả FP mây có conf cao mà PROTECT_CONF che).
    Ngưỡng theo CHÊNH LỆCH reflectance so với nền, nên offset baseline tự triệt tiêu.
    Tự bỏ qua với L2A (không có B10) hoặc khi thiếu band. Trả (keep_b, keep_s, drop_b, drop_s)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    z, zs = np.zeros((0, 4), np.float32), np.zeros((0,), np.float32)
    if not cfg.get("CLOUD_B10", True) or len(boxes) == 0:
        return boxes, scores, z, zs
    if detect_product_level(raster_path) == "L2A":
        return boxes, scores, z, zs                        # L2A không có B10
    g = _granule_root(raster_path)
    b10 = _find_s2_band(g, "B10", detect_product_level(raster_path)) if g else None
    if not b10:
        if verbose:
            print("[cloud] không có band B10 -> bỏ qua lọc cirrus")
        return boxes, scores, z, zs
    with rasterio.open(b10) as src:
        arr = src.read(1).astype(np.float32); b10_T = src.transform
    with rasterio.open(raster_path) as src:
        tci_T = src.transform
    valid = arr[arr > 0]
    bg = float(np.percentile(valid, cfg.get("CLOUD_B10_BG_PCT", 20))) if valid.size else 0.0
    excess = (_box_mean_values(arr, b10_T, boxes, tci_T) - bg) / 10000.0
    drop = excess >= float(cfg.get("CLOUD_B10_THR", 0.010))
    if verbose and bool(drop.any()):
        print(f"[cloud] B10 cirrus: bỏ {int(drop.sum())}/{len(boxes)} box "
              f"(chênh reflectance >= {cfg.get('CLOUD_B10_THR', 0.010)})")
    return boxes[~drop], scores[~drop], boxes[drop], scores[drop]


def _has_cloud_source(raster_path, cfg):
    """True nếu có nguồn mask THẬT kèm theo ảnh (SCL/CLDPRB/CLASSI/GML, hoặc đủ
    band cho OmniCloudMask). False = 'chỉ có ảnh TCI', không có gì để mask."""
    if str(cfg.get("CLOUD_SOURCE", "auto")).lower() == "omnicloudmask":
        g = _granule_root(raster_path)
        if g is not None:
            lvl = detect_product_level(raster_path)
            if (_find_s2_band(g, "B04", lvl) and _find_s2_band(g, "B03", lvl)
                    and (_find_s2_band(g, "B8A", lvl) or _find_s2_band(g, "B08", lvl))):
                return True
    kind, _ = find_cloud_source(raster_path, verbose=False)
    return kind != "brightness"


def get_cloud_mask_for(raster_path, cfg, verbose=True):
    """Trả CloudMask cho raster nếu bật CLOUD_MASK và mode != 'off', ngược lại None.
    Bọc lỗi để 1 ảnh hỏng mask không làm chết cả vòng lặp."""
    if not cfg.get("CLOUD_MASK", False) or str(cfg.get("CLOUD_MASK_MODE", "after")).lower() == "off":
        return None
    if cfg.get("CLOUD_SKIP_IF_TCI_ONLY", True) and not _has_cloud_source(raster_path, cfg):
        if verbose:
            print("[cloud] chỉ có ảnh TCI (không có band/mask kèm theo) -> bỏ qua lọc mây cho ảnh này")
        return None
    try:
        lvl = detect_product_level(raster_path, verbose=verbose)
        return build_cloud_mask(raster_path, cfg=cfg, verbose=verbose)
    except Exception as e:
        print("[cloud] không dựng được mask, bỏ qua lọc mây:", str(e).splitlines()[0])
        return None


def apply_cloud_filter(raster_path, boxes, scores, cloud_mask, cfg):
    """Lọc detections sau infer (mode 'after'). Trả (boxes, scores, n_dropped)."""
    if cloud_mask is None or str(cfg.get("CLOUD_MASK_MODE", "after")).lower() != "after":
        return boxes, scores, 0
    with rasterio.open(raster_path) as src:
        T = src.transform
    kb, ks, db, ds = filter_boxes_by_cloud(
        boxes, scores, cloud_mask, T,
        frac_thr=cfg.get("CLOUD_FRAC_THR", 0.35),
        mode=cfg.get("CLOUD_FILTER", "fraction"),
        protect_conf=cfg.get("CLOUD_PROTECT_CONF", 0.5))
    return kb, ks, int(len(db))


print("Loaded cloud-mask helpers.")

## 5. Tiền xử lý + suy luận theo cửa sổ trượt

Ba việc, bám sát repo gốc:

1. **Warp về EPSG:3857** ở đúng mức zoom mà mô hình được huấn luyện (`src/data/warp.py`:
   Sentinel-2 10m → zoom 13 → **9.5546 m/px**). Dùng `WarpedVRT` nên **không ghi file tạm và
   không nạp cả ảnh vào RAM**; các band khác (vd `B08`) bám đúng lưới của TCI nên tự khớp pixel.
   Band 16-bit bị **cắt ngưỡng ở 255** (`np.clip`) — đúng như `src/data/image.py` lúc train,
   *không* co giãn về 0–255.
2. **Cửa sổ trượt** `WINDOW=2048`, bỏ viền `PADDING=400`, rồi **NMS theo khoảng cách tâm**
   (`src/inference/pipeline.py::nms`) vì đây là mô hình *point detection*.
3. **Đổi toạ độ box về pixel ảnh gốc** (TCI ở UTM), để mọi bước sau — lọc mây, cắt crop,
   vẽ overview, xuất vector — chạy trên ảnh gốc.

Chạy **không có ảnh lịch sử (historical overlaps)**: backbone tự đắp *blank history*
(`SwinTransformerRCNNBackbone.forward`), đúng như lệnh inference "without historical overlaps"
trong README.

In [ ]:
import contextlib
import math

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.warp import calculate_default_transform, transform as rio_transform
from rasterio.windows import Window

DST_CRS = "EPSG:3857"
WEB_MERCATOR_M = 2 * math.pi * 6378137

# ---- đọc ảnh RGB để VẼ (trên raster GỐC, không phải ảnh đã warp) --------------


def _band_idx(nbands):
    return [1, 2, 3] if nbands >= 3 else [1, 1, 1]


def _to_uint8(arr):
    if arr.dtype == np.uint8:
        return arr
    a = arr.astype(np.float32)
    return np.clip(a / (float(a.max()) or 1.0) * 255.0, 0, 255).astype(np.uint8)


def _read_rgb_window(src, window):
    arr = src.read(indexes=_band_idx(src.count), window=window, boundless=True, fill_value=0)
    return _to_uint8(np.transpose(arr, (1, 2, 0)))


# ---- Tiền xử lý: warp về EPSG:3857 đúng như src/data/warp.py ------------------


def pick_web_mercator_res(in_pixel_size):
    """Chọn độ phân giải Web Mercator ở mức zoom ngay TRÊN độ phân giải gốc.

    Sao chép nguyên logic `src/data/warp.py::warp` (nhánh epsg:3857): duyệt zoom
    0..19, lấy mức đầu tiên có pixel < 1.1 * pixel gốc. Với Sentinel-2 10m ->
    zoom 13 -> 9.5546 m/px. Đây chính là lưới ảnh mà mô hình được huấn luyện.
    """
    out_px = None
    for zoom in range(20):
        out_px = WEB_MERCATOR_M / 512 / (2 ** zoom)
        if out_px < in_pixel_size * 1.1:
            break
    return out_px


def open_warped_stack(stack, paths, pixel_size=10.0):
    """Mở danh sách file thành các WarpedVRT DÙNG CHUNG một lưới EPSG:3857.

    Dùng WarpedVRT thay vì gdalwarp ra file tạm: không tốn đĩa, đọc theo cửa sổ
    (lazy) nên không phải nạp cả ảnh 10980x10980 vào RAM. File đầu tiên (TCI)
    định nghĩa lưới, các file sau bám đúng transform/width/height đó nên các band
    tự khớp pixel — thay cho bước dò lệch ±32px trong src/data/image.py.
    """
    grid, vrts = None, []
    for path in paths:
        src = stack.enter_context(rasterio.open(path))
        if src.crs is None:
            raise RuntimeError(f"{os.path.basename(path)} không có CRS -> không warp được")
        if grid is None:
            in_px = float(pixel_size or min(abs(src.transform.a), abs(src.transform.e)))
            out_px = pick_web_mercator_res(in_px)
            T, W, H = calculate_default_transform(
                src.crs, DST_CRS, src.width, src.height, *src.bounds,
                resolution=(out_px, out_px))
            grid = (T, int(W), int(H), out_px)
        T, W, H, out_px = grid
        vrts.append(stack.enter_context(WarpedVRT(
            src, crs=DST_CRS, transform=T, width=W, height=H,
            resampling=Resampling.bilinear)))
    return vrts, grid


def resolve_channel_paths(raster_path, channels, cfg, verbose=True):
    """Tìm file cho từng kênh mô hình cần. Kênh đầu ('tci') chính là ảnh đang xét."""
    level = detect_product_level(raster_path)
    granule = _granule_root(raster_path)
    paths, missing = [], []
    for ch in channels:
        name = ch["Name"].lower()
        if name == "tci":
            paths.append(raster_path)
            continue
        band = name.upper()                       # b08 -> B08
        hit = _find_s2_band(granule, band, level) if granule else None
        if hit:
            paths.append(hit)
        else:
            paths.append(None)
            missing.append(band)
    if missing and verbose:
        print(f"[prep] thiếu band {missing} cạnh ảnh -> "
              f"{'đắp 0' if cfg.get('MISSING_BAND', 'zeros') == 'zeros' else 'BỎ ảnh này'}")
    return paths, missing


def read_stack_window(vrts, counts, window):
    """Ghép các kênh thành mảng (C, h, w) uint8, tự đệm 0 ở phần tràn ra ngoài ảnh.

    `np.clip(..., 0, 255)` là bắt buộc để giống hệt tiền xử lý lúc train
    (src/data/image.py): band 16-bit như B08 bị CẮT NGƯỠNG ở 255, không hề
    được co giãn về 0-255.
    """
    r0, c0 = int(window.row_off), int(window.col_off)
    h, w = int(window.height), int(window.width)
    out = []
    for vrt, count in zip(vrts, counts):
        buf = np.zeros((count, h, w), np.uint8)
        if vrt is not None:
            cs, rs = max(0, c0), max(0, r0)
            ce, re_ = min(vrt.width, c0 + w), min(vrt.height, r0 + h)
            if ce > cs and re_ > rs:
                arr = vrt.read(window=Window(cs, rs, ce - cs, re_ - rs))
                arr = np.clip(arr, 0, 255).astype(np.uint8)
                if arr.shape[0] < count:
                    arr = np.repeat(arr[:1], count, axis=0)
                arr = arr[:count]
                buf[:, rs - r0: rs - r0 + arr.shape[1], cs - c0: cs - c0 + arr.shape[2]] = arr
        out.append(buf)
    return np.concatenate(out, axis=0)


# ---- Cửa sổ trượt + NMS ------------------------------------------------------


def _window_offsets(size, win, pad):
    """Vị trí cửa sổ, sao chép src/inference/pipeline.py::apply_model."""
    if size <= win:
        return [0]
    step = max(1, win - 2 * pad)
    offs = [0] + list(range(step, size - win, step)) + [size - win]
    return sorted({o for o in offs if 0 <= o <= size - win})


def distance_nms(boxes, scores, thresh):
    """NMS theo KHOẢNG CÁCH tâm — đúng cách repo lọc trùng (pipeline.py::nms).

    Mô hình là point-detector: hai box cách nhau < `thresh` px coi như cùng 1 tàu,
    giữ box điểm cao hơn.
    """
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    if len(boxes) == 0 or thresh is None or thresh <= 0:
        return np.arange(len(boxes))
    cx = (boxes[:, 0] + boxes[:, 2]) / 2.0
    cy = (boxes[:, 1] + boxes[:, 3]) / 2.0
    order = np.argsort(-scores)
    dead = np.zeros(len(order), bool)
    keep = []
    for i, idx in enumerate(order):
        if dead[i]:
            continue
        keep.append(int(idx))
        d2 = (cx[order] - cx[idx]) ** 2 + (cy[order] - cy[idx]) ** 2
        dead |= d2 <= float(thresh) ** 2
    return np.array(sorted(keep), dtype=int)


def detect_on_stack(vrts, counts, model, cfg, device, verbose=True):
    """Trượt cửa sổ trên ảnh đã warp, trả (boxes, scores) theo px của lưới 3857."""
    from torchvision.ops import nms as iou_nms

    win = int(cfg["WINDOW"])
    pad = int(cfg["PADDING"])
    ov = int(cfg["OVERLAP"])
    ref = next(v for v in vrts if v is not None)
    W, H = ref.width, ref.height
    xs = _window_offsets(W, win, pad)
    ys = _window_offsets(H, win, pad)
    if verbose:
        print(f"[infer] lưới 3857: {W}x{H}px -> {len(xs)}x{len(ys)} = {len(xs)*len(ys)} cửa sổ "
              f"(window={win}, padding={pad}, overlap={ov})")

    boxes_all, scores_all = [], []
    buf, buf_off = [], []
    n_used = 0

    def flush():
        if not buf:
            return
        with torch.no_grad():
            dets, _ = model(buf)
        for out, (ox, oy) in zip(dets, buf_off):
            b = out["boxes"].detach().cpu().numpy()
            s = out["scores"].detach().cpu().numpy()
            if len(b) == 0:
                continue
            cx = (b[:, 0] + b[:, 2]) / 2.0
            cy = (b[:, 1] + b[:, 3]) / 2.0
            # Chỉ giữ phát hiện nằm trong vùng lõi của cửa sổ (bỏ viền padding),
            # trừ khi đó là cạnh ngoài cùng của ảnh — hệt apply_model.
            kb = [pad, pad, win - pad, win - pad]
            if ox == 0:
                kb[0] = 0
            if oy == 0:
                kb[1] = 0
            if ox >= W - win:
                kb[2] = win
            if oy >= H - win:
                kb[3] = win
            keep = ((cx >= kb[0] - ov) & (cx <= kb[2] + ov) &
                    (cy >= kb[1] - ov) & (cy <= kb[3] + ov) & (s >= float(cfg["CONF"])))
            if not keep.any():
                continue
            b = b[keep].copy()
            b[:, [0, 2]] += ox
            b[:, [1, 3]] += oy
            boxes_all.append(b)
            scores_all.append(s[keep])
        buf.clear()
        buf_off.clear()

    for oy in ys:
        for ox in xs:
            arr = read_stack_window(vrts, counts, Window(ox, oy, win, win))
            if int(arr[:3].max()) < int(cfg["SKIP_DARK"]):
                continue                      # cửa sổ toàn nodata/đen -> bỏ cho nhanh
            buf.append(torch.as_tensor(arr).to(device).float() / 255)
            buf_off.append((ox, oy))
            n_used += 1
            if len(buf) >= int(cfg["BATCH"]):
                flush()
    flush()

    if verbose:
        print(f"[infer] {n_used} cửa sổ có dữ liệu")
    if not boxes_all:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32)

    boxes = np.concatenate(boxes_all).astype(np.float32)
    scores = np.concatenate(scores_all).astype(np.float32)
    if cfg.get("NMS_IOU", 0):
        k = iou_nms(torch.from_numpy(boxes), torch.from_numpy(scores),
                    float(cfg["NMS_IOU"])).numpy()
        boxes, scores = boxes[k], scores[k]
    k = distance_nms(boxes, scores, cfg.get("NMS_DIST", 10))
    boxes, scores = boxes[k], scores[k]
    order = np.argsort(-scores)
    if verbose:
        print(f"[infer] sau NMS: {len(boxes)} phát hiện (score >= {cfg['CONF']})")
    return boxes[order], scores[order]


def predict_attributes(vrts, counts, model, boxes, device, crop_size=120, batch=32):
    """Dự đoán dài/rộng/hướng/tốc độ/nghề cá cho từng phát hiện (pipeline.py::apply_model)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    cols = ["vessel_length_m", "vessel_width_m", "heading_deg", "vessel_speed_k",
            "is_fishing_vessel"]
    if model is None or len(boxes) == 0:
        return {c: np.zeros((len(boxes),), np.float32) for c in cols}
    ref = next(v for v in vrts if v is not None)
    W, H = ref.width, ref.height
    nch = int(getattr(model, "num_channels", sum(counts)))
    cx = np.clip(((boxes[:, 0] + boxes[:, 2]) / 2).astype(int), crop_size // 2, W - crop_size // 2)
    cy = np.clip(((boxes[:, 1] + boxes[:, 3]) / 2).astype(int), crop_size // 2, H - crop_size // 2)
    res = {c: np.zeros((len(boxes),), np.float32) for c in cols}
    with torch.no_grad():
        for start in range(0, len(boxes), batch):
            crops = []
            for i in range(start, min(start + batch, len(boxes))):
                arr = read_stack_window(
                    vrts, counts,
                    Window(int(cx[i]) - crop_size // 2, int(cy[i]) - crop_size // 2,
                           crop_size, crop_size))
                crops.append(torch.as_tensor(arr[:nch]).to(device).float() / 255)
            out = model(crops)[0].cpu()
            for j in range(len(crops)):
                i = start + j
                res["vessel_length_m"][i] = 100 * out[j, 0].item()
                res["vessel_width_m"][i] = 100 * out[j, 1].item()
                heading = torch.nn.functional.softmax(out[j, 2:18], dim=0)
                res["heading_deg"][i] = float(int(heading.argmax()) * (360.0 / 16))
                res["vessel_speed_k"][i] = out[j, 18].item()
                res["is_fishing_vessel"][i] = torch.nn.functional.softmax(
                    out[j, 19:21], dim=0)[1].item()
    return res


# ---- Đưa toạ độ từ lưới 3857 về pixel của ẢNH GỐC ----------------------------


def map_boxes_to_source(boxes, vrt_transform, src_path):
    """Box (px lưới 3857) -> box (px raster gốc, vd TCI ở UTM).

    Nhờ vậy mọi bước sau (lọc mây, cắt crop, vẽ overview, xuất vector) đều làm
    trên ảnh gốc — dùng lại y nguyên bộ helper của notebook YOLO.
    """
    boxes = np.asarray(boxes, np.float64).reshape(-1, 4)
    if len(boxes) == 0:
        return np.zeros((0, 4), np.float32)
    n = len(boxes)
    cols = np.concatenate([boxes[:, 0], boxes[:, 2], boxes[:, 0], boxes[:, 2]])
    rows = np.concatenate([boxes[:, 1], boxes[:, 1], boxes[:, 3], boxes[:, 3]])
    T = vrt_transform
    X = T.a * cols + T.b * rows + T.c
    Y = T.d * cols + T.e * rows + T.f
    with rasterio.open(src_path) as src:
        src_T, src_crs = src.transform, src.crs
    xs, ys = rio_transform(DST_CRS, src_crs, X.tolist(), Y.tolist())
    inv = ~src_T
    xs = np.asarray(xs, np.float64)
    ys = np.asarray(ys, np.float64)
    cc = (inv.a * xs + inv.b * ys + inv.c).reshape(4, n)
    rr = (inv.d * xs + inv.e * ys + inv.f).reshape(4, n)
    return np.stack([cc.min(0), rr.min(0), cc.max(0), rr.max(0)], axis=1).astype(np.float32)


def boxes_lonlat(boxes, src_path):
    """Tâm mỗi box (px ảnh gốc) -> (lon, lat) WGS84."""
    boxes = np.asarray(boxes, np.float64).reshape(-1, 4)
    if len(boxes) == 0:
        return np.zeros((0,), np.float64), np.zeros((0,), np.float64)
    with rasterio.open(src_path) as src:
        T, crs = src.transform, src.crs
    cx = (boxes[:, 0] + boxes[:, 2]) / 2.0
    cy = (boxes[:, 1] + boxes[:, 3]) / 2.0
    X = T.a * cx + T.b * cy + T.c
    Y = T.d * cx + T.e * cy + T.f
    lon, lat = rio_transform(crs, "EPSG:4326", X.tolist(), Y.tolist())
    return np.asarray(lon), np.asarray(lat)


def run_detector_on_raster(raster_path, cfg, verbose=True):
    """Toàn bộ chuỗi cho 1 ảnh: chọn band -> warp -> trượt cửa sổ -> thuộc tính
    -> đổi toạ độ về px ảnh gốc. Trả (boxes_src, scores, attrs) hoặc None nếu bỏ qua."""
    paths, missing = resolve_channel_paths(raster_path, DET_CHANNELS, cfg, verbose=verbose)
    if missing and str(cfg.get("MISSING_BAND", "zeros")).lower() == "skip":
        return None
    counts = [c["Count"] for c in DET_CHANNELS]
    with contextlib.ExitStack() as stack:
        present = [p for p in paths if p]
        vrts_present, grid = open_warped_stack(stack, present, pixel_size=cfg.get("S2_PIXEL_SIZE", 10.0))
        it = iter(vrts_present)
        vrts = [next(it) if p else None for p in paths]
        if verbose:
            print(f"[prep] warp -> EPSG:3857 @ {grid[3]:.4f} m/px, {grid[1]}x{grid[2]}px, "
                  f"kênh={[c['Name'] for c in DET_CHANNELS]}")
        boxes_w, scores = detect_on_stack(vrts, counts, detector, cfg, DEVICE, verbose=verbose)
        attrs = predict_attributes(vrts, counts, attr_model, boxes_w, DEVICE)
    boxes_src = map_boxes_to_source(boxes_w, grid[0], raster_path)
    return boxes_src, scores, attrs


print("Loaded inference helpers.")

## 6. Vẽ & xuất kết quả

In [ ]:
import json as _json

import cv2
import geopandas as gpd
import pandas as pd
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image
from shapely.geometry import box as shp_box


def draw_overview(raster_path, boxes=None, max_side=1600, title=None, save_path=None,
                  show=True, drop_boxes=None, cloud_mask=None):
    """Ảnh thu nhỏ toàn cảnh + khung dự đoán (đỏ) + khung bị loại do mây (cam nét đứt)."""
    boxes = np.zeros((0, 4), np.float32) if boxes is None else np.asarray(boxes)
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        scale = min(1.0, max_side / max(W, H))
        ow, oh = max(1, int(W * scale)), max(1, int(H * scale))
        ov = _to_uint8(np.transpose(
            src.read(indexes=_band_idx(src.count), out_shape=(3, oh, ow)), (1, 2, 0)))
        _T = src.transform
    fig, ax = plt.subplots(figsize=(13, max(3, 13 * oh / ow)))
    ax.imshow(ov)
    if cloud_mask is not None:
        try:
            ovT = _T * rasterio.Affine.scale(W / ow, H / oh)
            cmov = cloud_mask.window_mask(ovT, 0, 0, ow, oh)
            overlay = np.zeros((oh, ow, 4), np.float32)
            overlay[cmov] = (0.55, 0.55, 0.55, 0.35)
            ax.imshow(overlay)
        except Exception:
            pass
    if drop_boxes is not None and len(drop_boxes):
        for (x1, y1, x2, y2) in (np.asarray(drop_boxes) * scale):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                           edgecolor="orange", linewidth=0.8, linestyle="--"))
    for (x1, y1, x2, y2) in (np.asarray(boxes) * scale):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor="red", linewidth=0.8))
    handles = [Line2D([0], [0], color="red", lw=2, label="Dự đoán")]
    if drop_boxes is not None and len(drop_boxes):
        handles.append(Line2D([0], [0], color="orange", lw=2, linestyle="--", label="Bỏ (mây)"))
    ax.legend(handles=handles, loc="upper right", fontsize=9)
    ax.set_title(title or f"{os.path.basename(raster_path)} — {len(boxes)} dự đoán", fontsize=12)
    ax.axis("off")
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


def show_detection_crops(raster_path, boxes, scores, topk=8, pad=40, ncol=4,
                         save_path=None, show=True):
    if len(boxes) == 0:
        print("No vessel")
        return
    order = np.argsort(-scores)[:topk]
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        nrow = (len(order) + ncol - 1) // ncol
        fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow))
        axes = np.atleast_1d(axes).flatten()
        for ax, i in zip(axes, order):
            x1, y1, x2, y2 = boxes[i]
            cx0, cy0 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            cw = min(W - cx0, int(x2 - x1) + 2 * pad)
            ch = min(H - cy0, int(y2 - y1) + 2 * pad)
            ax.imshow(_read_rgb_window(src, Window(cx0, cy0, cw, ch)))
            ax.add_patch(patches.Rectangle((x1 - cx0, y1 - cy0), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="red", linewidth=1.0))
            ax.set_title(f"score={scores[i]:.2f}", fontsize=9)
            ax.axis("off")
        for ax in axes[len(order):]:
            ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


def _attr_frame(attrs, n):
    """Chuẩn hoá dict thuộc tính về đúng n dòng (rỗng nếu tắt attr model)."""
    if not attrs:
        return {}
    out = {}
    for k, v in attrs.items():
        v = np.asarray(v).reshape(-1)
        if len(v) == n and np.any(v):
            out[k] = [round(float(x), 3) for x in v]
    return out


def detections_to_vectors(raster_path, boxes, scores, attrs=None, out_geojson=None,
                          out_gpkg=None, out_points=None):
    """Xuất vector cho QGIS. GeoPackage giữ CRS gốc, GeoJSON luôn ở EPSG:4326."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    with rasterio.open(raster_path) as src:
        T, crs = src.transform, src.crs
    geoms, confs = [], []
    for (x1, y1, x2, y2), s in zip(boxes.tolist(), scores.tolist()):
        (X1, Y1) = rasterio.transform.xy(T, y1, x1)
        (X2, Y2) = rasterio.transform.xy(T, y2, x2)
        geoms.append(shp_box(min(X1, X2), min(Y1, Y2), max(X1, X2), max(Y1, Y2)))
        confs.append(round(float(s), 4))
    data = {"score": confs, "class": ["vessel"] * len(confs)}
    data.update(_attr_frame(attrs, len(confs)))
    gdf = gpd.GeoDataFrame(data, geometry=geoms, crs=crs)
    if len(gdf) == 0:
        print("  (0 phát hiện -> không ghi file vector)")
        return gdf
    if out_gpkg:
        gdf.to_file(out_gpkg, driver="GPKG")
        print("  gpkg   ", os.path.basename(out_gpkg))
    if out_geojson:
        (gdf.to_crs(4326) if crs is not None else gdf).to_file(out_geojson, driver="GeoJSON")
        print("  geojson", os.path.basename(out_geojson))
    if out_points:
        pts = gdf.copy()
        pts["geometry"] = [g.centroid for g in gdf.geometry]
        (pts.to_crs(4326) if crs is not None else pts).to_file(out_points, driver="GeoJSON")
        print("  points ", os.path.basename(out_points))
    return gdf


def save_predictions_csv(raster_path, boxes, scores, attrs, out_csv):
    """CSV theo tinh thần predictions.csv của repo (1 dòng = 1 phát hiện)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    lon, lat = boxes_lonlat(boxes, raster_path)
    stem = os.path.splitext(os.path.basename(raster_path))[0]
    df = pd.DataFrame({
        "detect_id": [f"{stem}_{i}" for i in range(len(boxes))],
        "scene_id": stem,
        "score": np.round(scores, 4),
        "lon": np.round(lon, 6),
        "lat": np.round(lat, 6),
        "column": ((boxes[:, 0] + boxes[:, 2]) / 2).astype(int) if len(boxes) else [],
        "row": ((boxes[:, 1] + boxes[:, 3]) / 2).astype(int) if len(boxes) else [],
    })
    for k, v in _attr_frame(attrs, len(boxes)).items():
        df[k] = v
    df.to_csv(out_csv, index=False)
    print("  csv    ", os.path.basename(out_csv))
    return df


def save_detections_json(raster_path, boxes, scores, out_json):
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    lon, lat = boxes_lonlat(boxes, raster_path)
    with rasterio.open(raster_path) as src:
        crs = src.crs
    recs = [{"score": round(float(s), 4),
             "bbox_xyxy_pixel": [round(float(v), 1) for v in b],
             "center_lonlat": [round(float(x), 6), round(float(y), 6)]}
            for b, s, x, y in zip(boxes.tolist(), scores.tolist(), lon, lat)]
    out = {"raster": os.path.basename(raster_path), "num_detections": len(recs),
           "crs": (str(crs) if crs is not None else None), "detections": recs}
    with open(out_json, "w") as f:
        _json.dump(out, f, indent=2)
    return out


def export_detection_crops(raster_path, boxes, scores, out_dir, pad=40, draw_box=True):
    """Mỗi phát hiện -> 1 ảnh PNG (sắp theo score giảm dần)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    if len(boxes) == 0:
        return 0
    os.makedirs(out_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(raster_path))[0]
    n = 0
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        for rank, i in enumerate(np.argsort(-scores)):
            x1, y1, x2, y2 = (float(v) for v in boxes[i])
            cx0, cy0 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            cw = min(W - cx0, int(x2 - x1) + 2 * pad)
            ch = min(H - cy0, int(y2 - y1) + 2 * pad)
            if cw <= 0 or ch <= 0:
                continue
            crop = _read_rgb_window(src, Window(cx0, cy0, cw, ch)).copy()
            if draw_box:
                cv2.rectangle(crop, (int(x1 - cx0), int(y1 - cy0)),
                              (int(x2 - cx0), int(y2 - cy0)), (255, 0, 0), 1)
            Image.fromarray(crop).save(
                os.path.join(out_dir, f"{stem}_{rank:03d}_score{float(scores[i]):.2f}.png"))
            n += 1
    return n


def export_hard_negatives(raster_path, neg_boxes, out_dir, tile=800):
    """Cắt tile quanh mỗi box nghi mây thành mẫu NEGATIVE (ảnh + nhãn rỗng, chuẩn YOLO)."""
    neg_boxes = np.asarray(neg_boxes, np.float32).reshape(-1, 4)
    if len(neg_boxes) == 0:
        return 0
    img_dir = os.path.join(out_dir, "images")
    lbl_dir = os.path.join(out_dir, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(raster_path))[0]
    n = 0
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        tw, th = min(tile, W), min(tile, H)
        for j, (x1, y1, x2, y2) in enumerate(neg_boxes):
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
            ox = min(max(0, int(round(cx - tw / 2))), max(0, W - tw))
            oy = min(max(0, int(round(cy - th / 2))), max(0, H - th))
            Image.fromarray(_read_rgb_window(src, Window(ox, oy, tw, th))).save(
                os.path.join(img_dir, f"{stem}_neg{j:03d}.png"))
            open(os.path.join(lbl_dir, f"{stem}_neg{j:03d}.txt"), "w").close()
            n += 1
    return n


def b10_cirrus_drop_mask(raster_path, boxes, cfg, verbose=True):
    """Mask True = box là cirrus/mây theo band B10 (1375nm) của L1C.

    Cùng cơ sở vật lý với `filter_boxes_by_b10_cirrus` ở ô mask mây (hơi nước hấp
    thụ 1375nm nên biển & tàu TỐI ở B10, mây/cirrus trên cao thì SÁNG), nhưng trả
    về MẶT NẠ để giữ được chỉ số hàng — cần thiết vì còn phải cắt theo cột thuộc tính.
    """
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    drop = np.zeros(len(boxes), bool)
    if not cfg.get("CLOUD_B10", True) or len(boxes) == 0:
        return drop
    level = detect_product_level(raster_path)
    if level == "L2A":
        return drop                                    # L2A không có B10
    granule = _granule_root(raster_path)
    b10 = _find_s2_band(granule, "B10", level) if granule else None
    if not b10:
        if verbose:
            print("[cloud] không có band B10 -> bỏ qua lọc cirrus")
        return drop
    with rasterio.open(b10) as src:
        arr = src.read(1).astype(np.float32)
        b10_T = src.transform
    with rasterio.open(raster_path) as src:
        tci_T = src.transform
    valid = arr[arr > 0]
    bg = float(np.percentile(valid, cfg.get("CLOUD_B10_BG_PCT", 20))) if valid.size else 0.0
    excess = (_box_mean_values(arr, b10_T, boxes, tci_T) - bg) / 10000.0
    drop = excess >= float(cfg.get("CLOUD_B10_THR", 0.010))
    if verbose and bool(drop.any()):
        print(f"[cloud] B10 cirrus: bỏ {int(drop.sum())}/{len(boxes)} box "
              f"(chênh reflectance >= {cfg.get('CLOUD_B10_THR', 0.010)})")
    return drop


print("Loaded export helpers.")

## 7. Tải ảnh từ Drive

Mỗi `.zip` tải từ Copernicus đã chứa đủ cấu trúc `.SAFE` (GRANULE/IMG_DATA/QI_DATA +
`MTD_*.xml`) — cần cho band `B08` và mask mây, nên hãy để nguyên, đừng chỉ up mỗi file TCI.

In [ ]:
import gdown, glob, zipfile

RASTER_EXTS = (".jp2", ".tif", ".tiff", ".png", ".jpg", ".jpeg")


def extract_zips(root, delete=False):
    """Giải nén mọi *.zip trong thư mục tải về. Mỗi zip tải từ Copernicus đã chứa
    sẵn cấu trúc đầy đủ của 1 ảnh Sentinel-2 (folder .SAFE với GRANULE/IMG_DATA/
    QI_DATA + MTD_*.xml) — đây cũng là thứ cần để lấy band B08 và mask mây.
    Idempotent: bỏ qua nếu đã giải nén từ lần chạy trước."""
    zips = sorted(glob.glob(os.path.join(root, "**", "*.zip"), recursive=True))
    for z in zips:
        dest = os.path.dirname(z)
        try:
            with zipfile.ZipFile(z) as zf:
                tops = sorted({n.split("/", 1)[0] for n in zf.namelist() if n.strip("/")})
                if tops and all(os.path.exists(os.path.join(dest, t)) for t in tops):
                    print(f"  [zip] đã giải nén trước đó, bỏ qua: {os.path.basename(z)}")
                else:
                    print(f"  [zip] giải nén {os.path.basename(z)} -> {tops[:1]} ...")
                    zf.extractall(dest)
            if delete:
                os.remove(z)
                print(f"  [zip] đã xoá {os.path.basename(z)}")
        except zipfile.BadZipFile:
            print(f"  [zip] file hỏng, bỏ qua: {os.path.basename(z)}")
        except Exception as e:
            print(f"  [zip] lỗi {os.path.basename(z)}: {str(e).splitlines()[0]}")
    return len(zips)


def select_inference_rasters(root):
    """Chọn ảnh để chạy detect: giữ file TCI, bỏ band/mask phụ trợ (chúng chỉ là
    đầu vào cho B08 / mask mây). Nếu bộ dữ liệu không có TCI thì giữ ảnh rời."""
    all_files = [p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
                 if p.lower().endswith(RASTER_EXTS)]
    has_tci = any(is_tci(p) for p in all_files)
    rasters = []
    for p in all_files:
        if is_tci(p):
            rasters.append(p)
        elif is_s2_band_or_mask(p):
            continue
        elif not has_tci:
            rasters.append(p)
    rasters = sorted(set(rasters))
    return rasters, {p: find_safe_root(p) for p in rasters}


# ---- chạy ----
gdown.download_folder(CFG["DRIVE_FOLDER_URL"], output=CFG["DL_DIR"], quiet=False, use_cookies=False)

n_zip = extract_zips(CFG["DL_DIR"], delete=CFG.get("UNZIP_DELETE_ZIP", False))
if n_zip:
    print(f"Giải nén xong {n_zip} file .zip.")

raster_files, SAFE_ROOTS = select_inference_rasters(CFG["DL_DIR"])
if CFG["LIMIT"]:
    raster_files = raster_files[:CFG["LIMIT"]]

print(f"\nTải xong. Có {len(raster_files)} ảnh sẽ được suy luận:")
for p in raster_files:
    lvl = detect_product_level(p, safe_root=SAFE_ROOTS.get(p)) or "?"
    tag = "SAFE" if SAFE_ROOTS.get(p) else "loose"
    print(f"  - [{lvl:>3}/{tag}] {os.path.relpath(p, CFG['DL_DIR'])}  ({os.path.getsize(p) / 1e6:.1f} MB)")
assert raster_files, "Không tìm thấy ảnh nào (TCI hoặc ảnh rời). Kiểm tra lại zip/thư mục Drive."

## 8. Chạy & xuất kết quả

In [ ]:
import shutil
import time

import pandas as pd
from IPython.display import display

CLOUD_ON = CFG.get("CLOUD_MASK", False) and str(CFG.get("CLOUD_MASK_MODE", "after")).lower() != "off"
if CLOUD_ON and str(CFG.get("CLOUD_MASK_MODE", "after")).lower() != "after":
    print("[cloud] notebook này chỉ hỗ trợ mode 'after' -> tự chuyển sang 'after'")
    CFG["CLOUD_MASK_MODE"] = "after"

CROPS_DIR = os.path.join(CFG["OUT_DIR"], "crops")
if CFG.get("SAVE_CROPS_ZIP", False):
    shutil.rmtree(CROPS_DIR, ignore_errors=True)
    os.makedirs(CROPS_DIR, exist_ok=True)

HARDNEG_DIR = os.path.join(CFG["OUT_DIR"], "hard_negatives")
if CFG.get("SAVE_HARD_NEG", False):
    shutil.rmtree(HARDNEG_DIR, ignore_errors=True)
    os.makedirs(HARDNEG_DIR, exist_ok=True)

summary = []
for p in raster_files:
    stem = os.path.splitext(os.path.basename(p))[0]
    print("\n" + "=" * 70 + f"\n>>> {os.path.basename(p)}")
    t0 = time.time()

    # --- suy luận: warp -> cửa sổ trượt Faster R-CNN -> toạ độ về px ảnh gốc ---
    try:
        res = run_detector_on_raster(p, CFG, verbose=True)
    except Exception as e:
        print("Lỗi suy luận:", str(e).splitlines()[0])
        summary.append({"file": os.path.basename(p), "num_ships": None, "cloud_dropped": None,
                        "seconds": None, "size_MB": round(os.path.getsize(p) / 1e6, 1)})
        continue
    if res is None:
        print("Bỏ qua ảnh này (thiếu band mà mô hình yêu cầu).")
        summary.append({"file": os.path.basename(p), "num_ships": None, "cloud_dropped": None,
                        "seconds": None, "size_MB": round(os.path.getsize(p) / 1e6, 1)})
        continue
    b, s, attrs = res

    # bản THÔ trước lọc mây (để đối chiếu trong QGIS)
    raw_b = np.asarray(b, np.float32).reshape(-1, 4).copy()
    raw_s = np.asarray(s, np.float32).reshape(-1).copy()
    keep_idx = np.arange(len(raw_b))

    # --- lọc mây (mode 'after') ---
    cloud_mask = get_cloud_mask_for(p, CFG, verbose=True) if CLOUD_ON else None
    n_cloud = 0
    drop_boxes = None
    dropped_all = np.zeros((0, 4), np.float32)
    if cloud_mask is not None and len(b):
        with rasterio.open(p) as _src:
            _T = _src.transform
        frac = cloud_mask.box_cloud_fraction(
            b if CFG.get("CLOUD_FILTER", "fraction") != "center"
            else np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2,
                           (b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2], 1), _T)
        drop = frac >= float(CFG.get("CLOUD_FRAC_THR", 0.35))
        pc = float(CFG.get("CLOUD_PROTECT_CONF", 0.0) or 0.0)
        if pc > 0:
            drop = drop & (s < pc)
        n_cloud = int(drop.sum())
        dropped_all = np.concatenate([dropped_all, b[drop]], 0)
        drop_boxes = b[drop] if CFG.get("DRAW_CLOUD", True) else None
        keep_idx = keep_idx[~drop]
        b, s = b[~drop], s[~drop]
        if n_cloud:
            print(f"[cloud] bỏ {n_cloud} dự đoán trùng vùng mây -> còn {len(b)}")

    # --- lọc cirrus bằng band B10 (chỉ L1C) — bắt cả FP mây có score cao ---
    if CFG.get("CLOUD_B10", True) and len(b):
        drop2 = b10_cirrus_drop_mask(p, b, CFG, verbose=True)
        if drop2.any():
            n_cloud += int(drop2.sum())
            dropped_all = np.concatenate([dropped_all, b[drop2]], 0)
            if CFG.get("DRAW_CLOUD", True):
                drop_boxes = (b[drop2] if drop_boxes is None
                              else np.concatenate([np.asarray(drop_boxes), b[drop2]], 0))
            keep_idx = keep_idx[~drop2]
            b, s = b[~drop2], s[~drop2]
            print(f"[cloud] sau lọc B10 còn {len(b)} dự đoán")

    attrs_keep = {k: np.asarray(v)[keep_idx] for k, v in (attrs or {}).items()}

    # ---------------- XUẤT KẾT QUẢ (không có GeoTIFF/JP2 annotated) ----------------
    if CFG.get("SAVE_VECTOR", True):
        detections_to_vectors(
            p, b, s, attrs_keep,
            out_geojson=os.path.join(CFG["OUT_DIR"], f"{stem}_pred.geojson"),
            out_gpkg=os.path.join(CFG["OUT_DIR"], f"{stem}_pred.gpkg"),
            out_points=(os.path.join(CFG["OUT_DIR"], f"{stem}_pred_points.geojson")
                        if CFG.get("SAVE_POINTS", True) else None))
        if CFG.get("SAVE_VECTOR_RAW", True):
            detections_to_vectors(
                p, raw_b, raw_s, attrs,
                out_geojson=os.path.join(CFG["OUT_DIR"], f"{stem}_pred_raw.geojson"),
                out_gpkg=os.path.join(CFG["OUT_DIR"], f"{stem}_pred_raw.gpkg"))
    if CFG.get("SAVE_CSV", True):
        save_predictions_csv(p, b, s, attrs_keep,
                             os.path.join(CFG["OUT_DIR"], f"{stem}_predictions.csv"))
    if CFG.get("SAVE_JSON", True):
        save_detections_json(p, b, s, os.path.join(CFG["OUT_DIR"], f"{stem}_detections.json"))

    if CFG.get("SAVE_OVERVIEW", True):
        parts = [f"{len(b)} dự đoán (score≥{CFG['CONF']})"]
        if n_cloud:
            parts.append(f"{n_cloud} bỏ do mây")
        draw_overview(p, boxes=b, drop_boxes=drop_boxes,
                      cloud_mask=(cloud_mask if CFG.get("DRAW_CLOUD", True) else None),
                      max_side=CFG["OVERVIEW_MAX"], title=f"{stem} — " + " · ".join(parts),
                      save_path=os.path.join(CFG["OUT_DIR"], f"{stem}_overview.png"))

    if len(b):
        if CFG.get("SAVE_CROPS_PNG", True):
            show_detection_crops(p, b, s, topk=8,
                                 save_path=os.path.join(CFG["OUT_DIR"], f"{stem}_crops.png"))
        if CFG.get("SAVE_CROPS_ZIP", False):
            crop_sub = os.path.join(CROPS_DIR, stem)
            shutil.rmtree(crop_sub, ignore_errors=True)
            os.makedirs(crop_sub, exist_ok=True)
            n_saved = export_detection_crops(p, b, s, crop_sub, pad=CFG.get("CROPS_PAD", 40),
                                             draw_box=CFG.get("CROPS_DRAW_BOX", True))
            if n_saved:
                shutil.make_archive(os.path.join(CFG["OUT_DIR"], f"{stem}_crops"), "zip", crop_sub)
                print(f"[crops] {n_saved} crop -> {stem}_crops.zip")

    if CFG.get("SAVE_HARD_NEG", False):
        sel = str(CFG.get("HARD_NEG_SOURCE", "dropped")).lower()
        if sel == "kept":
            neg = np.asarray(b, np.float32).reshape(-1, 4)
        elif sel == "all":
            neg = np.concatenate([np.asarray(b, np.float32).reshape(-1, 4), dropped_all], 0)
        else:
            neg = dropped_all
        n_neg = export_hard_negatives(p, neg, HARDNEG_DIR, tile=CFG.get("HARD_NEG_TILE", 800))
        if n_neg:
            print(f"[hard-neg] {n_neg} tile ({sel}) -> hard_negatives/")

    summary.append({"file": os.path.basename(p), "num_ships": int(len(b)),
                    "cloud_dropped": (n_cloud if (cloud_mask is not None or CFG.get("CLOUD_B10", True)) else None),
                    "seconds": round(time.time() - t0, 1),
                    "size_MB": round(os.path.getsize(p) / 1e6, 1)})

if CFG.get("SAVE_HARD_NEG", False):
    _imgs = os.path.join(HARDNEG_DIR, "images")
    if os.path.isdir(_imgs) and any(os.scandir(_imgs)):
        n = len([f for f in os.listdir(_imgs) if f.endswith(".png")])
        shutil.make_archive(os.path.join(CFG["OUT_DIR"], "hard_negatives"), "zip", HARDNEG_DIR)
        print(f"\nHard-negative: {n} tile -> hard_negatives.zip")

print("\nSUMMARY")
df = pd.DataFrame(summary)
display(df)
df.to_csv(os.path.join(CFG["OUT_DIR"], "summary.csv"), index=False)
print("\nKết quả lưu tại:", CFG["OUT_DIR"])
print("Kéo *_pred.geojson (hoặc *_pred_points.geojson) vào QGIS là xem được ngay.")

---

## Ghi chú

**Ngưỡng `CONF`.** Faster R-CNN ở repo này cho điểm cao hơn YOLO nhiều; `src/main.py` mặc định
`conf=0.9`. Nếu bắt thiếu tàu thì hạ dần (0.7 → 0.5) và xem lại `{stem}_pred_raw.geojson`.

**Tốc độ.** README của repo báo ~2–3 phút/ảnh trên GPU và ~30 phút/ảnh trên CPU. Nếu hết VRAM,
giảm `BATCH` về 1 hoặc `WINDOW` về 1024 (`PADDING` khoảng 200).

**Thuộc tính tàu.** Đặt `POSTPROCESS_MODEL_DIR` trỏ tới artifact `attr/...` để có thêm các cột
`vessel_length_m`, `vessel_width_m`, `heading_deg`, `vessel_speed_k`, `is_fishing_vessel` trong
GeoJSON/CSV. Repo yêu cầu detector và attr model **dùng chung bộ kênh gốc**
(`src/data/image.py::prepare_scenes`), ví dụ cả hai đều `tci + b08`.

**Thiếu band.** Nếu mô hình cần `b08` mà thư mục chỉ có TCI rời, `MISSING_BAND="zeros"` sẽ đắp 0
cho band thiếu (kết quả kém hơn); đặt `"skip"` để bỏ hẳn ảnh đó.